# 학습용 프로젝트: 표준 ML 파이프라인 한 바퀴 돌기

**목표: "머신러닝 프로젝트가 어떤 흐름으로 돌아가는가"를 끝까지 한 번 경험하기.**

본 과제(심정지 조기경보)의 데이터는 양성이 1% 남짓이라 무엇을 해도 지표가 바닥에 붙어 있어
**배우기에는 최악의 데이터**입니다. 그래서 여기서는 신호가 뚜렷한 데이터로 흐름을 익히고,
거기서 얻은 감각을 본 과제에 옮깁니다.

## 데이터: Pima Indians Diabetes

| 항목 | 값 |
|---|---|
| 크기 | 768명 × 8개 피처 |
| 예측 대상 | 5년 내 당뇨 발병 여부 (0/1) |
| 양성 비율 | **34.9%** |
| 특징 | 결측이 `0`으로 위장돼 있음 (실전 정제 연습) |

**왜 이 데이터인가**
- **의료 도메인** — 배운 내용이 본 과제로 그대로 전이됩니다.
- **양성 34.9%** — 지표를 해석할 수 있습니다. (1%짜리는 뭘 해도 0.02라 판단이 안 됨)
- **768행** — 경북대 실제 데이터(573명)와 규모가 비슷합니다. "소표본에서 뭘 조심해야 하나"까지 배웁니다.
- **숨은 결측** — 실제 임상 데이터에서 늘 만나는 문제입니다.

## 진행 순서

| 단계 | 내용 | 핵심 개념 |
|---|---|---|
| 1 | 데이터 구조 파악 | dtype, 결측, 타깃 분포 |
| 2 | EDA 시각화 | 분포·상관·타깃별 차이 |
| 3 | 전처리 + 분할 | **데이터 누수(leakage)** |
| 4 | Feature Engineering | 도메인 지식 → 새 변수 |
| 5 | 베이스라인 3종 | LightGBM / XGBoost / CatBoost |
| 6 | 하이퍼파라미터 튜닝 | Optuna, 교차검증 |
| 7 | 앙상블 | soft voting |
| 8 | 최종 평가 + 해석 | ROC/PR, 혼동행렬, SHAP |

> 플롯 제목은 **영어**입니다 — 서버에 한글 폰트가 없으면 matplotlib에서 □로 깨지기 때문입니다.
> 설명은 이 마크다운 셀들에 한글로 있습니다.

## 0. 준비

필요 패키지: `lightgbm`, `xgboost`, `catboost`, `optuna`, `shap`.
없으면 아래 셀이 알려줍니다. 설치: `pip install lightgbm xgboost catboost optuna shap`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (9, 5)

RANDOM_STATE = 42          # 재현성: 모든 난수 고정
N_TRIALS = 40              # Optuna 탐색 횟수 (늘릴수록 오래 걸림)
np.random.seed(RANDOM_STATE)

# 설치 여부 확인
available = {}
for name in ["lightgbm", "xgboost", "catboost", "optuna", "shap"]:
    try:
        __import__(name)
        available[name] = True
    except ImportError:
        available[name] = False
print("패키지 상태:", available)
missing = [k for k, v in available.items() if not v]
if missing:
    print(f"\n설치 필요: pip install {' '.join(missing)}")

## 1. 데이터 로드 및 구조 분석

**가장 먼저 할 일은 모델링이 아니라 "이 데이터가 뭔지" 파악하는 것**입니다. 순서:

1. **크기** — 행(샘플) × 열(피처). 행이 적으면 복잡한 모델은 과적합합니다.
2. **자료형** — 숫자형/범주형 구분.
3. **결측** — 얼마나, 어디에 비었나.
4. **타깃 분포** — 양성 비율. 이게 모든 지표 해석의 **기준선**이 됩니다.

In [ ]:
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv"
COLS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
        "Insulin", "BMI", "DiabetesPedigree", "Age", "Outcome"]

df = pd.read_csv(URL, header=None, names=COLS)

print(f"크기: {df.shape[0]}행 × {df.shape[1]}열")
print(f"\n타깃 분포:")
print(df["Outcome"].value_counts().rename({0: "음성(당뇨X)", 1: "양성(당뇨O)"}))
print(f"\n>>> 양성 비율 = {df['Outcome'].mean():.1%}  <- 이 값을 기억하세요. 모든 지표의 기준선입니다.")
df.head()

In [ ]:
# 자료형과 기술통계
print("=== 자료형 ===")
print(df.dtypes)
print("\n=== 기술통계 ===")
df.describe().T.round(2)

### ⚠️ 위 표에서 이상한 점을 찾으셨나요?

`Glucose`(혈당), `BloodPressure`(혈압), `SkinThickness`(피부두께), `Insulin`(인슐린), `BMI`의
**최솟값이 0**입니다. **살아있는 사람의 혈압이 0일 수는 없습니다.**

즉 이 데이터는 **결측을 `0`으로 기록**해 놓았습니다. `df.isna()`로는 절대 안 잡힙니다.

> **실전 교훈**: `describe()`의 min/max를 반드시 확인하세요. 생리학적으로 불가능한 값은
> 결측이거나 오류입니다. 이걸 놓치고 학습하면 "혈압 0인 환자" 수십 명을 모델이 진짜로 믿습니다.
> — 본 과제에서도 실제 MIMIC 데이터의 맥박 0, 화씨/섭씨 혼용을 이렇게 발견했습니다.

In [ ]:
# 0이 결측인 컬럼들 (Pregnancies=0, Outcome=0은 정상이므로 제외)
ZERO_IS_MISSING = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

miss = pd.DataFrame({
    "0의 개수": (df[ZERO_IS_MISSING] == 0).sum(),
    "비율": (df[ZERO_IS_MISSING] == 0).mean(),
})
print(miss.assign(비율=lambda d: (d["비율"] * 100).round(1).astype(str) + "%"))

fig, ax = plt.subplots(figsize=(8, 4))
(miss["비율"] * 100).sort_values().plot.barh(ax=ax, color="indianred")
ax.set_title("Hidden missing values (recorded as 0)")
ax.set_xlabel("% missing")
for i, v in enumerate(sorted(miss["비율"] * 100)):
    ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()

**Insulin의 49%, SkinThickness의 30%가 결측**입니다. 이 정도면 선택지가 갈립니다:
- 대치(imputation)한다 → 정보는 살리되 노이즈가 섞임
- 컬럼을 버린다 → 깨끗하지만 정보 손실
- **"결측이라는 사실" 자체를 피처로 만든다** → 뒤 4단계에서 해봅니다 (검사를 안 했다는 것 자체가 정보!)

## 2. EDA 시각화

숫자표만으로는 감이 안 옵니다. 그림으로 봅니다.

**무엇을 보려는 건가**: "이 피처가 타깃을 구분하는 데 쓸모가 있나?"
→ 양성군과 음성군의 분포가 **겹치지 않을수록** 그 피처는 유용합니다.

In [ ]:
# 0을 NaN으로 바꾼 사본으로 시각화 (0이 분포를 왜곡하므로)
viz = df.copy()
viz[ZERO_IS_MISSING] = viz[ZERO_IS_MISSING].replace(0, np.nan)

features = [c for c in df.columns if c != "Outcome"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), features):
    for label, color, name in [(0, "steelblue", "negative"), (1, "crimson", "positive")]:
        ax.hist(viz.loc[viz.Outcome == label, col].dropna(), bins=25,
                alpha=0.6, color=color, label=name, density=True)
    ax.set_title(col, fontsize=11)
    ax.set_yticks([])
axes[0, 0].legend(fontsize=9)
fig.suptitle("Feature distributions by outcome (separated = useful feature)", fontsize=13)
plt.tight_layout(); plt.show()

**읽는 법**: 빨강(양성)과 파랑(음성) 봉우리가 **옆으로 밀려 있으면** 그 피처는 판별력이 있습니다.
`Glucose`를 보세요 — 양성군이 확연히 오른쪽입니다. 반면 `BloodPressure`는 거의 겹칩니다.

이건 예상과 맞습니다(혈당이 당뇨의 직접 지표). **도메인 지식과 데이터가 일치하는지 확인**하는 것이
EDA의 핵심입니다. 어긋나면 데이터 오류이거나 새로운 발견입니다.

In [ ]:
# 타깃과의 상관계수 — 어떤 피처가 얼마나 관련 있나
corr_target = viz.corr()["Outcome"].drop("Outcome").sort_values()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
colors = ["crimson" if v > 0 else "steelblue" for v in corr_target]
corr_target.plot.barh(ax=ax[0], color=colors)
ax[0].set_title("Correlation with outcome")
ax[0].axvline(0, color="k", lw=0.8)

im = ax[1].imshow(viz.corr(), cmap="RdBu_r", vmin=-1, vmax=1)
ax[1].set_xticks(range(len(viz.columns)))
ax[1].set_xticklabels(viz.columns, rotation=90, fontsize=8)
ax[1].set_yticks(range(len(viz.columns)))
ax[1].set_yticklabels(viz.columns, fontsize=8)
ax[1].set_title("Correlation matrix")
plt.colorbar(im, ax=ax[1], shrink=0.8)
plt.tight_layout(); plt.show()

print("타깃과 상관이 높은 순:")
print(corr_target.abs().sort_values(ascending=False).round(3))

**주의**: 상관계수는 **선형 관계**만 잡습니다. 상관이 0이어도 U자형 관계가 있을 수 있어요.
트리 기반 모델(LightGBM 등)은 비선형도 잡으니, 상관 낮다고 피처를 버리지 마세요.

**오른쪽 히트맵**에서 피처끼리 진하게 붉은 쌍(예: `Age`–`Pregnancies`)은 **다중공선성**입니다.
선형모델에선 문제지만 트리 모델은 견딥니다.

## 3. 전처리와 데이터 분할 — **여기가 제일 중요합니다**

### 왜 분할을 먼저 하는가

모델 성능을 정직하게 측정하려면 **한 번도 본 적 없는 데이터**로 평가해야 합니다.
그래서 데이터를 학습용(train)과 평가용(test)으로 나눕니다.

### 데이터 누수(Data Leakage) — 초보가 가장 많이 하는 실수

결측을 평균으로 채운다고 합시다. **전체 데이터의 평균**으로 채우면?
→ test 데이터의 정보가 train에 스며듭니다. 이러면 성능이 실제보다 좋게 나오고,
실전 배포하면 무너집니다.

**규칙: 모든 통계량(평균/표준편차/중앙값)은 train에서만 계산해서 test에 적용한다.**

### stratify를 쓰는 이유
양성 34.9%를 train/test 양쪽에 **동일 비율로** 유지합니다. 안 하면 우연히 test에 양성이
몰려서 평가가 왜곡됩니다.

In [ ]:
from sklearn.model_selection import train_test_split

# 1) 0 -> NaN (진짜 결측으로 표시)
data = df.copy()
data[ZERO_IS_MISSING] = data[ZERO_IS_MISSING].replace(0, np.nan)

X = data.drop(columns="Outcome")
y = data["Outcome"]

# 2) 분할 먼저! (전처리보다 먼저)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"train: {X_train.shape[0]}명 (양성 {y_train.mean():.1%})")
print(f"test : {X_test.shape[0]}명 (양성 {y_test.mean():.1%})")
print("\n-> stratify 덕분에 양성 비율이 양쪽 동일합니다.")

In [ ]:
# 3) 결측 대치 — 중앙값을 **train에서만** 계산
medians = X_train.median()          # <- train만 사용 (누수 방지)
print("train 중앙값 (이 값으로 양쪽을 채웁니다):")
print(medians.round(2))

X_train_f = X_train.fillna(medians)
X_test_f = X_test.fillna(medians)   # <- test 자신의 중앙값이 아님!

print(f"\n결측 남은 개수: train={X_train_f.isna().sum().sum()}, test={X_test_f.isna().sum().sum()}")

## 4. Feature Engineering

**원본 피처를 조합해 "모델이 배우기 쉬운 형태"의 새 변수를 만드는 작업.**
좋은 피처 하나가 모델 튜닝 100번보다 낫습니다.

만들 피처와 그 근거:

| 새 피처 | 계산 | 도메인 근거 |
|---|---|---|
| `Glucose_BMI` | 혈당 × BMI | 고혈당 + 비만이 겹치면 위험이 곱으로 증가 |
| `Insulin_Glucose_ratio` | 인슐린 / 혈당 | **인슐린 저항성** 지표 (당뇨의 기전) |
| `BMI_category` | 저체중/정상/과체중/비만 | 임상에서 실제로 쓰는 구간 |
| `Age_group` | 연령대 구간 | 위험이 나이에 선형이 아님 |
| `is_*_missing` | 결측 여부 (0/1) | **검사를 안 했다는 사실 자체가 정보** |

마지막 항목이 중요합니다. 인슐린 검사를 안 받은 환자는 "의사가 필요없다고 판단했다"는
뜻일 수 있어요. 결측 패턴이 정보를 담는 이런 경우를 **informative missingness**라 합니다.

In [ ]:
def add_features(X_raw, X_filled):
    """원본(결측 정보 보존)과 대치본을 받아 파생 피처를 만든다."""
    out = X_filled.copy()

    # 상호작용 / 비율
    out["Glucose_BMI"] = out["Glucose"] * out["BMI"]
    out["Insulin_Glucose_ratio"] = out["Insulin"] / out["Glucose"]
    out["Glucose_per_Age"] = out["Glucose"] / out["Age"]

    # 임상 구간 (범주 -> 숫자 코드)
    out["BMI_category"] = pd.cut(out["BMI"], bins=[0, 18.5, 25, 30, 100],
                                 labels=[0, 1, 2, 3]).astype(int)
    out["Age_group"] = pd.cut(out["Age"], bins=[0, 30, 40, 50, 100],
                              labels=[0, 1, 2, 3]).astype(int)

    # informative missingness — 원본에서 결측이었는지
    for col in ["Insulin", "SkinThickness", "BloodPressure"]:
        out[f"is_{col}_missing"] = X_raw[col].isna().astype(int)

    return out

X_train_fe = add_features(X_train, X_train_f)
X_test_fe = add_features(X_test, X_test_f)

print(f"피처 개수: {X_train_f.shape[1]}개 -> {X_train_fe.shape[1]}개")
print("\n추가된 피처:", [c for c in X_train_fe.columns if c not in X_train_f.columns])
X_train_fe.head(3)

## 5. 평가 지표 — **숫자를 읽는 법**

모델을 돌리기 전에 **무엇을 보고 좋고 나쁨을 판단할지** 정해야 합니다.

| 지표 | 의미 | 기준선 | 방향 |
|---|---|---|---|
| **Accuracy** | 전체 정답률 | 65.1% (전부 음성이라 찍기) | ↑ |
| **ROC-AUC** | 양성/음성 순위를 잘 매기는가 | 0.5 (찍기) | ↑ |
| **AUPRC** | 정밀도-재현율 곡선 아래 면적 | **0.349 (= 양성 비율)** | ↑ |
| **F1** | 정밀도와 재현율의 조화평균 | — | ↑ |

### ⚠️ Accuracy의 함정
전부 "음성"이라 찍어도 **65.1%** 나옵니다. 그러니 "정확도 70%!"는 거의 의미가 없어요.
불균형 데이터일수록 Accuracy는 무용합니다.

### AUPRC의 기준선은 양성 비율
이게 핵심입니다. **AUPRC는 절대값으로 읽으면 안 되고, 양성 비율과 비교**해야 합니다.
- 여기서는 양성 34.9% → 아무 정보 없는 모델의 AUPRC ≈ **0.349**
- AUPRC 0.70이 나왔다면 → 기준선의 2배 = 훌륭함
- (본 과제에서 AUPRC 0.027이 나왔던 건 기준선이 0.012라 겨우 2배였던 겁니다)

### 교차검증(Cross-Validation)을 쓰는 이유
768행은 적습니다. 한 번의 train/test 분할 결과는 **운에 크게 좌우**됩니다.
5-fold CV는 5번 나눠 평균 내므로 훨씬 안정적이고, **표준편차로 불확실성**까지 볼 수 있습니다.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             f1_score, precision_score, recall_score)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
BASELINE_AUPRC = y_train.mean()

def cv_score(model, X, y, name=""):
    """5-fold CV로 ROC-AUC와 AUPRC를 재고 평균±표준편차를 돌려준다."""
    res = cross_validate(model, X, y, cv=CV,
                         scoring=["roc_auc", "average_precision"], n_jobs=1)
    return {
        "model": name,
        "ROC-AUC": res["test_roc_auc"].mean(),
        "ROC_std": res["test_roc_auc"].std(),
        "AUPRC": res["test_average_precision"].mean(),
        "AUPRC_std": res["test_average_precision"].std(),
    }

print(f"AUPRC 기준선(양성 비율) = {BASELINE_AUPRC:.3f}")
print("-> 앞으로 나오는 AUPRC는 이 값과 비교해서 읽으세요.")

## 6. 베이스라인 모델 구축

**항상 단순한 모델부터 시작합니다.** 복잡한 모델이 단순한 모델을 못 이기면 쓸 이유가 없어요.

| 모델 | 특징 |
|---|---|
| **로지스틱 회귀** | 가장 단순. "이것보다 나은가?"의 기준 |
| **LightGBM** | 빠름. 대용량·고차원에 강함 |
| **XGBoost** | 가장 검증됨. 캐글 표준 |
| **CatBoost** | 범주형 처리 우수. 기본값으로도 잘 나옴 |

셋 다 **Gradient Boosting** 계열입니다 — 약한 트리를 순차적으로 쌓아 이전 트리의 오차를
보정하는 방식. **표(tabular) 데이터에서는 딥러닝보다 강한 것이 정설**입니다
(Grinsztajn et al., NeurIPS 2022).

### 피처 엔지니어링이 실제로 도움이 됐는지도 여기서 확인합니다
같은 모델을 **FE 전 / FE 후** 데이터로 각각 돌려 비교합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {}
models["LogisticRegression"] = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)

if available["lightgbm"]:
    from lightgbm import LGBMClassifier
    models["LightGBM"] = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, n_jobs=1)

if available["xgboost"]:
    from xgboost import XGBClassifier
    models["XGBoost"] = XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0, n_jobs=1
    )

if available["catboost"]:
    from catboost import CatBoostClassifier
    models["CatBoost"] = CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, thread_count=1)

print("사용할 모델:", list(models))

In [ ]:
# FE 전 / FE 후 비교
rows = []
for name, mdl in models.items():
    rows.append({**cv_score(mdl, X_train_f, y_train, name), "features": "original (8)"})
    rows.append({**cv_score(mdl, X_train_fe, y_train, name), "features": f"engineered ({X_train_fe.shape[1]})"})

baseline_df = pd.DataFrame(rows)
pivot = baseline_df.pivot(index="model", columns="features", values="AUPRC").round(4)
pivot["개선"] = (pivot.iloc[:, 0] - pivot.iloc[:, 1]).abs().round(4)
print("=== AUPRC (기준선 %.3f) ===" % BASELINE_AUPRC)
print(pivot.to_string())
print()
baseline_df.round(4).to_string(index=False)

In [ ]:
# 시각화: 모델별 · FE 전후 AUPRC
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

p = baseline_df.pivot(index="model", columns="features", values="AUPRC")
p.plot.bar(ax=ax[0], rot=20)
ax[0].axhline(BASELINE_AUPRC, color="red", ls="--", label=f"baseline {BASELINE_AUPRC:.3f}")
ax[0].set_title("AUPRC: original vs engineered features")
ax[0].set_ylabel("AUPRC"); ax[0].legend(fontsize=8)

# 오차막대로 CV 불확실성 표시
eng = baseline_df[baseline_df["features"].str.startswith("engineered")]
ax[1].barh(eng["model"], eng["AUPRC"], xerr=eng["AUPRC_std"],
           color="steelblue", capsize=4)
ax[1].axvline(BASELINE_AUPRC, color="red", ls="--", label="baseline")
ax[1].set_title("AUPRC with CV std (engineered features)")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### 결과를 정직하게 읽기 — 세 가지 함정

위 표를 **기대에 맞춰 읽지 말고 있는 그대로** 보세요. 자주 나타나는 상황:

**① 단순한 모델이 이길 수 있다**
로지스틱 회귀가 부스팅 3종을 이겼나요? 데이터가 작고(614행) 피처가 적고(8개) 관계가
대체로 선형이면 **충분히 일어나는 일**입니다. 부스팅은 데이터가 많고 상호작용이 복잡할 때
빛을 발합니다. → **"복잡한 모델이 당연히 낫다"는 착각을 버리세요.** 단순 모델을 항상 같이 돌려
비교하는 이유가 이것입니다.

**② Feature Engineering이 도움이 안 될 수도 있다**
`개선` 열이 작거나 음수인 모델이 있나요? 놀랄 일이 아닙니다. **트리 모델은 이미 분기를 통해
상호작용을 스스로 학습**합니다. `Glucose x BMI` 같은 곱을 명시적으로 넣어줘도
트리 입장에선 새 정보가 아니고, 오히려 피처가 16개로 늘어 **탐색 공간만 커져** 손해일 수 있어요.
(반면 선형모델은 곱셈을 표현할 수 없으므로 FE의 덕을 크게 봅니다 — 표에서 확인하세요.)

> **교훈**: FE는 "모델이 스스로 표현하지 못하는 것"을 넣어줄 때 효과가 있습니다.
> 무작정 피처를 늘리는 건 도움이 안 됩니다.

**③ 차이가 오차범위 안이면 "차이 없음"이다**

오른쪽 그래프의 **가로 오차막대가 CV 표준편차**입니다. 모델 간 차이가 이 막대보다 작다면
**우연히 이긴 것**일 수 있습니다.

768행처럼 데이터가 적으면 CV 표준편차가 커서, 소수점 셋째 자리 차이는 의미가 없습니다.
**"LightGBM이 XGBoost보다 0.003 높다 → LightGBM 승리"는 잘못된 결론**입니다.

> 이건 본 과제에도 그대로 적용됩니다. 경북대 데이터도 573명이라 소표본이에요.
> 모델 비교는 반드시 CV + 표준편차로 하고, 차이가 작으면 "차이 없음"이라고 정직하게 써야 합니다.

## 7. 하이퍼파라미터 튜닝 (Optuna)

**하이퍼파라미터**는 모델이 학습으로 알아내지 못하고 사람이 정해줘야 하는 값입니다
(트리 깊이, 학습률, 트리 개수 등).

**Optuna**는 이걸 자동으로 탐색합니다. 랜덤 서치와 달리 **이전 시도 결과를 보고 유망한 영역을
집중 탐색**(TPE 알고리즘)하므로 훨씬 효율적입니다.

### 중요: 탐색도 CV로
한 번의 분할 점수로 튜닝하면 그 분할에 과적합됩니다. **CV 평균**을 최적화 대상으로 삼습니다.

### 무엇을 최적화하나
**AUPRC**입니다. 정확도가 아니라. 불균형 데이터에서 의미 있는 지표를 골라야 합니다.

In [ ]:
if available["optuna"] and available["lightgbm"]:
    import optuna
    from lightgbm import LGBMClassifier
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves":       trial.suggest_int("num_leaves", 8, 64),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "subsample":        trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10, log=True),
        }
        mdl = LGBMClassifier(**params, random_state=RANDOM_STATE, verbose=-1, n_jobs=1)
        res = cross_validate(mdl, X_train_fe, y_train, cv=CV,
                             scoring="average_precision", n_jobs=1)
        return res["test_score"].mean()

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    mask = (baseline_df["model"] == "LightGBM") & baseline_df["features"].str.startswith("engineered")
    lgbm_default = baseline_df.loc[mask, "AUPRC"].iloc[0]
    print(f"기본 LightGBM AUPRC : {lgbm_default:.4f}")
    print(f"튜닝 후 최고 AUPRC  : {study.best_value:.4f}")
    print(f"개선폭              : {study.best_value - lgbm_default:+.4f}")
    print(f"\n최적 하이퍼파라미터:")
    for k, v in study.best_params.items():
        print(f"  {k:20s} = {v}")
else:
    study = None
    print("optuna 또는 lightgbm 미설치 — 이 단계를 건너뜁니다.")

In [ ]:
if study is not None:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

    # 탐색 과정: 시도별 점수와 누적 최고
    vals = [t.value for t in study.trials if t.value is not None]
    ax[0].plot(vals, "o", ms=4, alpha=0.5, label="each trial")
    ax[0].plot(np.maximum.accumulate(vals), "-", color="crimson", lw=2, label="best so far")
    ax[0].set_xlabel("trial"); ax[0].set_ylabel("CV AUPRC")
    ax[0].set_title("Optuna search progress"); ax[0].legend(fontsize=8)

    # 어떤 하이퍼파라미터가 중요했나
    try:
        imp = optuna.importance.get_param_importances(study)
        pd.Series(imp).sort_values().plot.barh(ax=ax[1], color="teal")
        ax[1].set_title("Hyperparameter importance")
    except Exception as e:
        ax[1].text(0.5, 0.5, f"importance unavailable\n{e}", ha="center")
    plt.tight_layout(); plt.show()

**왼쪽 그래프 읽는 법**: 빨간 선(누적 최고)이 **평평해지면 충분히 탐색한 것**입니다.
계속 오르고 있다면 trial을 늘릴 가치가 있습니다.

**튜닝으로 얼마나 오르나?** 보통 **1~3%** 정도입니다. Feature Engineering이 주는 개선보다
작은 경우가 많아요. **그래서 순서가 중요합니다: 좋은 피처 먼저, 튜닝은 마지막.**

---

### 💡 실전 함정: `n_jobs=-1`이 항상 빠른 게 아니다

이 노트북의 모든 `cross_validate`와 모델에 **`n_jobs=1`**을 명시했습니다. 이유:

`cross_validate(n_jobs=-1)`은 CPU 코어 수만큼 프로세스를 띄웁니다. 그런데 LightGBM/XGBoost도
**각자 내부에서 모든 코어로 멀티스레딩**합니다. 둘이 겹치면 4코어에 16개 스레드가 몰려
서로 코어를 뺏는 **oversubscription**이 발생합니다.

실제 측정값 (4코어, 이 데이터 기준):

| 설정 | 1 trial 소요 |
|---|---|
| `cross_validate(n_jobs=-1)`, 모델 기본 | **232.5초** |
| `cross_validate(n_jobs=1)`, `model(n_jobs=1)` | **0.34초** |

**680배 차이**입니다. 데이터가 작을 때는 병렬화 오버헤드가 계산량보다 크므로,
**직렬 실행이 압도적으로 빠릅니다.** "느리면 일단 n_jobs=-1"은 틀린 처방일 수 있어요.

## 8. 앙상블

**서로 다른 모델의 예측을 합치면 대체로 각각보다 낫습니다.**
모델마다 다른 실수를 하기 때문에, 평균 내면 실수가 상쇄됩니다.

- **Hard voting**: 다수결 (0/1 투표)
- **Soft voting**: **확률을 평균** ← 보통 이게 더 좋습니다. 확신의 정도까지 반영하니까요.

앙상블이 효과를 보려면 모델들이 **서로 달라야** 합니다. 똑같은 실수를 하는 모델을 모으면
의미가 없어요.

### 앙상블이 항상 좋은 것은 아니다

soft voting은 **모든 모델의 확률을 동등하게 평균**합니다. 그래서 성능이 낮은 모델이 섞이면
**최고 단일 모델보다 나빠질 수 있습니다.** 아래 표에서 앙상블이 1등인지 반드시 확인하세요.

앙상블이 최고 단일 모델보다 낮다면 선택지는:
- 약한 모델을 빼고 다시 구성
- 가중 평균(`weights=[...]`)으로 강한 모델에 더 큰 비중
- 그냥 **최고 단일 모델을 쓴다** ← 부끄러운 선택이 아닙니다

**"앙상블을 했으니 더 좋을 것"이라 가정하지 말고 반드시 측정하세요.**

In [ ]:
from sklearn.ensemble import VotingClassifier

# 튜닝된 LightGBM을 포함해 최종 후보 구성
final_models = {}
# 단순 모델도 반드시 후보에 넣는다 (베이스라인에서 가장 강했을 수도 있으므로)
final_models["LogisticRegression"] = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
if available["lightgbm"]:
    from lightgbm import LGBMClassifier
    best_params = study.best_params if study is not None else {}
    final_models["LightGBM(tuned)"] = LGBMClassifier(
        **best_params, random_state=RANDOM_STATE, verbose=-1, n_jobs=1)
if available["xgboost"]:
    from xgboost import XGBClassifier
    final_models["XGBoost"] = XGBClassifier(
        random_state=RANDOM_STATE, eval_metric="logloss", verbosity=0, n_jobs=1)
if available["catboost"]:
    from catboost import CatBoostClassifier
    final_models["CatBoost"] = CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, thread_count=1)

ensemble = VotingClassifier(
    estimators=[(k, v) for k, v in final_models.items()],
    voting="soft"
)

rows = [cv_score(m, X_train_fe, y_train, n) for n, m in final_models.items()]
rows.append(cv_score(ensemble, X_train_fe, y_train, "Ensemble(soft)"))
final_cv = pd.DataFrame(rows).sort_values("AUPRC", ascending=False)
print("=== 최종 후보 CV 성능 (기준선 %.3f) ===" % BASELINE_AUPRC)
print(final_cv.round(4).to_string(index=False))

## 9. 최종 평가 — **여기서 처음으로 test 데이터를 씁니다**

지금까지는 train 안에서 CV만 했습니다. test는 **단 한 번, 마지막에** 씁니다.
test를 보고 모델을 고치면 그 순간 test는 더 이상 "본 적 없는 데이터"가 아니게 됩니다.

In [ ]:
# CV에서 가장 좋았던 모델을 선택해 전체 train으로 학습 -> test 평가
best_name = final_cv.iloc[0]["model"]
best_model = ensemble if best_name.startswith("Ensemble") else final_models[best_name]
print(f"선택된 모델: {best_name}")

best_model.fit(X_train_fe, y_train)
proba = best_model.predict_proba(X_test_fe)[:, 1]
pred = (proba >= 0.5).astype(int)

print(f"\n=== TEST 성능 ===")
print(f"  Accuracy : {accuracy_score(y_test, pred):.4f}   (전부 음성 찍기 = {1 - y_test.mean():.4f})")
print(f"  ROC-AUC  : {roc_auc_score(y_test, proba):.4f}   (찍기 = 0.5)")
print(f"  AUPRC    : {average_precision_score(y_test, proba):.4f}   (기준선 = {y_test.mean():.4f})")
print(f"  F1       : {f1_score(y_test, pred):.4f}")
print(f"  Precision: {precision_score(y_test, pred):.4f}  (알람이 울렸을 때 진짜일 확률)")
print(f"  Recall   : {recall_score(y_test, pred):.4f}  (실제 양성 중 잡아낸 비율)")

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))

# ROC
fpr, tpr, _ = roc_curve(y_test, proba)
ax[0].plot(fpr, tpr, lw=2, label=f"AUC={roc_auc_score(y_test, proba):.3f}")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.4, label="random")
ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate")
ax[0].set_title("ROC curve"); ax[0].legend()

# PR
prec, rec, _ = precision_recall_curve(y_test, proba)
ax[1].plot(rec, prec, lw=2, color="darkorange",
           label=f"AUPRC={average_precision_score(y_test, proba):.3f}")
ax[1].axhline(y_test.mean(), color="red", ls="--", label=f"baseline={y_test.mean():.3f}")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].set_title("Precision-Recall curve"); ax[1].legend()

# Confusion matrix
cm = confusion_matrix(y_test, pred)
im = ax[2].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax[2].text(j, i, cm[i, j], ha="center", va="center", fontsize=16,
                   color="white" if cm[i, j] > cm.max() / 2 else "black")
ax[2].set_xticks([0, 1]); ax[2].set_xticklabels(["pred 0", "pred 1"])
ax[2].set_yticks([0, 1]); ax[2].set_yticklabels(["true 0", "true 1"])
ax[2].set_title("Confusion matrix")
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"정답 음성 {tn}명 | 오경보 {fp}명 | 놓친 환자 {fn}명 | 정답 양성 {tp}명")
print(f"\n-> 놓친 환자({fn}명)를 줄이려면 임계값을 낮추면 되지만, 오경보({fp}명)가 늘어납니다.")
print("   이 트레이드오프가 본 과제(심정지 조기경보)의 핵심 주제입니다.")

### 임계값(threshold)이라는 조절 손잡이

모델은 **확률**을 출력합니다. 0.5를 넘으면 양성이라 판정한 것뿐이고, 이 기준은 **바꿀 수 있습니다.**

- 임계값 ↓ → 더 많이 양성 판정 → **놓치는 환자 감소**, 오경보 증가
- 임계값 ↑ → 더 보수적 → 오경보 감소, **놓치는 환자 증가**

의료에서는 "놓치는 것"이 훨씬 위험하므로 보통 임계값을 낮게 잡습니다. 다만 너무 낮추면
오경보가 폭증해 **alarm fatigue**가 옵니다. — **이 균형점을 찾는 것이 본 과제의 주제**입니다.

In [ ]:
# 임계값을 바꿔가며 지표가 어떻게 변하는지
ths = np.linspace(0.05, 0.95, 91)
prec_l, rec_l, f1_l = [], [], []
for t in ths:
    p = (proba >= t).astype(int)
    prec_l.append(precision_score(y_test, p, zero_division=0))
    rec_l.append(recall_score(y_test, p))
    f1_l.append(f1_score(y_test, p, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ths, prec_l, label="Precision (알람의 정확도)")
ax.plot(ths, rec_l, label="Recall (놓치지 않는 비율)")
ax.plot(ths, f1_l, label="F1", ls="--")
best_t = ths[int(np.argmax(f1_l))]
ax.axvline(best_t, color="gray", ls=":", label=f"best F1 @ {best_t:.2f}")
ax.axvline(0.5, color="red", ls=":", alpha=0.5, label="default 0.5")
ax.set_xlabel("threshold"); ax.set_ylabel("score")
ax.set_title("Threshold trade-off"); ax.legend()
plt.tight_layout(); plt.show()

print(f"기본 임계값 0.5의 F1 = {f1_score(y_test, (proba>=0.5).astype(int)):.4f}")
print(f"최적 임계값 {best_t:.2f}의 F1 = {max(f1_l):.4f}")

## 10. 모델 해석 — "왜 그렇게 판단했나"

성능이 좋아도 **근거를 설명 못 하면 임상에서 쓰이지 않습니다.** 두 가지 방법:

1. **Feature Importance** — 모델 전체에서 어떤 피처가 많이 쓰였나 (전역적)
2. **SHAP** — 개별 환자의 예측에 각 피처가 얼마나 기여했나 (국소적) ← 본 과제의 핵심 기법

In [ ]:
# Feature importance
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X_train_fe.columns)
elif hasattr(best_model, "estimators_"):
    imps = [e.feature_importances_ for e in best_model.estimators_
            if hasattr(e, "feature_importances_")]
    imp = pd.Series(np.mean(imps, axis=0), index=X_train_fe.columns)
else:
    imp = None

if imp is not None:
    fig, ax = plt.subplots(figsize=(9, 6))
    imp.sort_values().plot.barh(ax=ax, color="darkgreen")
    ax.set_title("Feature importance")
    plt.tight_layout(); plt.show()
    print("상위 5개 피처:")
    print(imp.sort_values(ascending=False).head().round(1))

In [ ]:
# SHAP — 개별 예측의 근거
if available["shap"]:
    import shap
    model_for_shap = (final_models.get("LightGBM(tuned)")
                      or final_models.get("XGBoost")
                      or list(final_models.values())[0])
    model_for_shap.fit(X_train_fe, y_train)

    explainer = shap.TreeExplainer(model_for_shap)
    sv = explainer.shap_values(X_test_fe)
    if isinstance(sv, list):
        sv = sv[1]

    shap.summary_plot(sv, X_test_fe, show=False, plot_size=(9, 6))
    plt.title("SHAP summary — feature impact on prediction")
    plt.tight_layout(); plt.show()
else:
    print("shap 미설치 — 건너뜁니다.")

**SHAP summary plot 읽는 법**
- **세로축**: 피처 (위일수록 영향력 큼)
- **가로축**: 그 피처가 예측을 양성 쪽(오른쪽)/음성 쪽(왼쪽)으로 얼마나 밀었나
- **색**: 그 피처의 값 (빨강=높음, 파랑=낮음)

예: `Glucose`가 맨 위이고 빨간 점들이 오른쪽에 몰려 있다면
→ **"혈당이 높으면 당뇨 위험이 올라간다"** 를 모델이 학습했다는 뜻. 의학 상식과 일치하죠.

**모델이 상식과 어긋나게 학습했다면 데이터에 문제가 있는 겁니다.** SHAP은 성능 지표가
못 잡는 이런 오류를 잡아냅니다.

## 11. 단계별 기여도 정리

**어느 단계가 실제로 성능을 올렸는지** 한눈에 봅니다.
**막대가 내려가는 구간이 있다면 그게 "효과가 없었던 단계"** 입니다 — 그것도 훌륭한 결과입니다.

In [ ]:
orig = baseline_df[baseline_df.features.str.startswith("original")]
eng = baseline_df[baseline_df.features.str.startswith("engineered")]

stages = {}
stages["1. LogisticRegression\n(단순 기준)"] = orig[orig.model == "LogisticRegression"]["AUPRC"].iloc[0]
if available["lightgbm"]:
    stages["2. LightGBM\n(원본 피처)"] = orig[orig.model == "LightGBM"]["AUPRC"].iloc[0]
    stages["3. + Feature Eng."] = eng[eng.model == "LightGBM"]["AUPRC"].iloc[0]
    if study is not None:
        stages["4. + 튜닝"] = study.best_value
stages["5. + 앙상블"] = final_cv[final_cv.model.str.startswith("Ensemble")]["AUPRC"].iloc[0]

s = pd.Series(stages)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(s)), s.values, color="steelblue")
bars[int(np.argmax(s.values))].set_color("crimson")
ax.axhline(BASELINE_AUPRC, color="red", ls="--", label=f"no-skill baseline {BASELINE_AUPRC:.3f}")
ax.set_xticks(range(len(s))); ax.set_xticklabels(s.index, fontsize=9)
ax.set_ylabel("CV AUPRC"); ax.set_title("Effect of each stage (may go down!)")
for i, v in enumerate(s.values):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()

print(s.round(4).to_string())
print(f"\n기준선(no-skill {BASELINE_AUPRC:.3f}) 대비 최고 성능: {s.max() / BASELINE_AUPRC:.2f}배")

delta = s.diff().dropna()
print("\n단계별 변화량 (음수 = 오히려 나빠짐):")
for k, v in delta.items():
    print(f"  {k.replace(chr(10), ' '):30s} {v:+.4f}  {'개선' if v > 0 else '악화'}")

## 12. 정리 — 배운 것과 본 과제 적용

### 이 흐름이 표준입니다

```
데이터 이해 → EDA → 분할(누수 방지) → 전처리 → Feature Engineering
   → 베이스라인 여러 개 → 튜닝 → 앙상블 → 최종 test 평가 → 해석(SHAP)
```

### 꼭 기억할 원칙 6가지

1. **지표는 기준선과 비교해서 읽는다.** AUPRC의 기준선은 양성 비율입니다.
   절대값 0.03이 나쁜 게 아니라, 기준선이 0.012면 2.5배로 좋은 겁니다.
2. **Accuracy는 불균형 데이터에서 무의미하다.** 찍어도 65% 나옵니다.
3. **누수를 막아라.** 모든 통계량은 train에서만 계산합니다.
4. **CV 표준편차를 함께 보라.** 차이가 오차막대보다 작으면 "차이 없음"입니다.
5. **순서가 중요하다.** 좋은 피처 > 튜닝. FE에 시간을 더 쓰세요.
6. **test는 마지막 한 번만.**

### 본 과제(심정지 조기경보)에 적용하면

| 여기서 배운 것 | 본 과제에서 |
|---|---|
| 숨은 결측(0) 찾기 | MIMIC의 맥박 0, 화씨/섭씨 혼용 → `sanitize_vitals()` |
| informative missingness | 활력징후 측정 빈도 자체가 중증도 신호 |
| AUPRC 기준선 비교 | 양성 1.2% → 기준선 0.012와 비교해서 읽기 |
| 임계값 트레이드오프 | **오경보 vs 놓친 환자** = 프로젝트의 핵심 주제 |
| CV 표준편차 | 573명 소표본 → 모델 차이를 과장하지 않기 |
| SHAP 해석 | "왜 위험한가"를 임상의에게 제시 |

### 다음에 해볼 것

- `RANDOM_STATE`를 바꿔 돌려보기 → 결과가 얼마나 흔들리는지 체감 (소표본의 불안정성)
- FE 셀에서 피처를 추가/삭제 → 어떤 피처가 진짜 기여하는지
- Optuna `n_trials`를 100으로 → 더 오래 탐색하면 얼마나 더 오르는지
- 임계값을 0.3으로 → 놓친 환자와 오경보가 어떻게 바뀌는지